# DS1/GSE103224 — post-QC Scrublet doublet diagnostic

This notebook reproduces the original post-QC Scrublet check for
DS1/GSE103224.

Important interpretation:

- Scrublet is run on the complete post-QC DS1 object, matching the original
  analysis notebook.
- The resulting `doublet_score` and `predicted_doublet` fields are retained
  as diagnostic metadata.
- Predicted doublets are **not removed**, so the analytical cell count
  remains 23,207.
- This notebook therefore documents a doublet-screening diagnostic rather
  than an additional dataset-filtering step.

Input:
`data/processed/DS1_GSE103224/GSE103224_postQC.h5ad`

Output:
`data/processed/DS1_GSE103224/GSE103224_postQC_scrublet_checked.h5ad`

> **AnnData 0.12 compatibility patch:** backed `AnnData` objects are
> explicitly closed with `check.file.close()` rather than used as
> context managers.

In [1]:
import inspect
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

print("Python:", sys.version)
print("scanpy:", sc.__version__)

Python: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:20:11) [MSC v.1938 64 bit (AMD64)]
scanpy: 1.11.5


C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_18544\2866864517.py:15: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy:", sc.__version__)


In [2]:
def find_repository_root(start: Path) -> Path:
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start


PROJECT_ROOT = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", find_repository_root(Path.cwd()))
).expanduser().resolve()

INPUT_H5AD = Path(
    os.environ.get(
        "GSE103224_POSTQC_H5AD",
        PROJECT_ROOT
        / "data"
        / "processed"
        / "DS1_GSE103224"
        / "GSE103224_postQC.h5ad",
    )
).expanduser().resolve()

OUTPUT_H5AD = Path(
    os.environ.get(
        "GSE103224_SCRUBLET_H5AD",
        PROJECT_ROOT
        / "data"
        / "processed"
        / "DS1_GSE103224"
        / "GSE103224_postQC_scrublet_checked.h5ad",
    )
).expanduser().resolve()

AUDIT_DIR = (
    PROJECT_ROOT
    / "results"
    / "qc"
    / "DS1_GSE103224"
    / "00c_doublet_diagnostic"
)
FIGURE_DIR = AUDIT_DIR / "figures"

OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_INPUT_SHAPE = (23_207, 29_457)
EXPECTED_ORIGINAL_PREDICTED_DOUBLETS = 1

# These values are the standard defaults used by the original
# sc.pp.scrublet(adata) call. Supported arguments are detected at runtime
# to retain compatibility across nearby Scanpy versions.
REQUESTED_SCRUBLET_PARAMETERS = {
    "sim_doublet_ratio": 2.0,
    "expected_doublet_rate": 0.05,
    "n_prin_comps": 30,
    "random_state": 0,
}

print("Input:", INPUT_H5AD)
print("Output:", OUTPUT_H5AD)

if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        f"Post-QC input not found: {INPUT_H5AD}\n"
        "Run 00b_ds1_quality_control.ipynb first or set "
        "GSE103224_POSTQC_H5AD."
    )

Input: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas\data\processed\DS1_GSE103224\GSE103224_postQC.h5ad
Output: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas\data\processed\DS1_GSE103224\GSE103224_postQC_scrublet_checked.h5ad


In [3]:
def sparse_values(matrix):
    return matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()


def assert_raw_nonnegative_integer_counts(matrix, label: str):
    values = sparse_values(matrix)
    if values.size == 0:
        raise ValueError(f"{label} has no nonzero count values.")
    if not np.isfinite(values).all():
        raise ValueError(f"{label} contains non-finite values.")
    if np.any(values < 0):
        raise ValueError(f"{label} contains negative values.")
    if not np.equal(values, np.floor(values)).all():
        raise ValueError(f"{label} is not an integer raw-count matrix.")

In [4]:
adata = sc.read_h5ad(INPUT_H5AD)

if adata.shape != EXPECTED_INPUT_SHAPE:
    raise AssertionError(
        f"Expected input shape {EXPECTED_INPUT_SHAPE}, observed {adata.shape}"
    )
if "sample_id" not in adata.obs:
    raise KeyError("sample_id is missing from adata.obs.")
if "counts" not in adata.layers:
    raise KeyError("The post-QC object is missing layers['counts'].")

assert_raw_nonnegative_integer_counts(adata.X, "adata.X")
assert_raw_nonnegative_integer_counts(
    adata.layers["counts"], 'adata.layers["counts"]'
)

# Make the notebook safe to rerun on an in-memory object.
for column in ("doublet_score", "predicted_doublet"):
    if column in adata.obs:
        del adata.obs[column]

print(adata)

AnnData object with n_obs × n_vars = 23207 × 29457
    obs: 'sample_id', 'batch', 'geo_sample_id', 'geo_id', 'core_dataset', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb'
    var: 'ensembl_id_version', 'gene_symbol', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
    uns: 'build_timestamp', 'dataset_id', 'dataset_level_QC', 'geo_accession', 'source_files'
    layers: 'counts'


In [5]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse


# ------------------------------------------------------------------
# Validate Scrublet input
# ------------------------------------------------------------------
assert adata.n_obs > 0, "adata contains no cells."
assert adata.n_vars > 0, "adata contains no genes."

assert adata.obs_names.is_unique, (
    "Cell identifiers in adata.obs_names are not unique."
)

assert adata.var_names.is_unique, (
    "Gene identifiers in adata.var_names are not unique."
)

if not sparse.issparse(adata.X):
    raise TypeError(
        "adata.X must be a SciPy sparse count matrix before Scrublet. "
        f"Observed type: {type(adata.X)}"
    )

# CSR is generally the appropriate sparse orientation for cell-wise
# processing. This does not densify the matrix.
if not sparse.isspmatrix_csr(adata.X):
    adata.X = adata.X.tocsr()

adata.X.eliminate_zeros()

assert np.isfinite(adata.X.data).all(), (
    "adata.X contains NaN or infinite values."
)

assert np.all(adata.X.data >= 0), (
    "Scrublet input contains negative values."
)


# ------------------------------------------------------------------
# Memory-safe and deterministic Scrublet parameters
# ------------------------------------------------------------------
scrublet_parameters = {
    "batch_key": None,
    "sim_doublet_ratio": 2.0,
    "expected_doublet_rate": 0.05,
    "stdev_doublet_rate": 0.02,
    "synthetic_doublet_umi_subsampling": 1.0,
    "knn_dist_metric": "euclidean",
    "normalize_variance": True,
    "log_transform": False,

    # Critical runtime patch:
    # keep dimensionality reduction sparse-compatible.
    "mean_center": False,

    "n_prin_comps": 30,
    "use_approx_neighbors": True,
    "get_doublet_neighbor_parents": False,
    "n_neighbors": None,
    "threshold": None,
    "verbose": True,
    "copy": False,
    "random_state": 0,
}

assert scrublet_parameters["mean_center"] is False
assert scrublet_parameters["normalize_variance"] is True
assert scrublet_parameters["copy"] is False

print("Scrublet input shape:", adata.shape)
print("Scrublet input matrix:", type(adata.X).__name__)
print("Scrublet matrix dtype:", adata.X.dtype)
print("Scrublet matrix nonzero entries:", int(adata.X.nnz))
print("Scrublet parameters used:")
for parameter, value in scrublet_parameters.items():
    print(f"  {parameter}: {value}")


# ------------------------------------------------------------------
# Run Scrublet
# ------------------------------------------------------------------
sc.pp.scrublet(
    adata,
    **scrublet_parameters,
)


# ------------------------------------------------------------------
# Validate Scrublet results
# ------------------------------------------------------------------
required_scrublet_columns = [
    "doublet_score",
    "predicted_doublet",
]

missing_scrublet_columns = [
    column
    for column in required_scrublet_columns
    if column not in adata.obs.columns
]

assert not missing_scrublet_columns, (
    "Scrublet did not create the expected columns: "
    f"{missing_scrublet_columns}"
)

adata.obs["doublet_score"] = pd.to_numeric(
    adata.obs["doublet_score"],
    errors="raise",
).astype("float64")

adata.obs["predicted_doublet"] = (
    adata.obs["predicted_doublet"]
    .astype(bool)
)

assert adata.obs["doublet_score"].notna().all(), (
    "Scrublet generated missing doublet scores."
)

assert np.isfinite(
    adata.obs["doublet_score"].to_numpy(dtype=np.float64)
).all(), (
    "Scrublet generated non-finite doublet scores."
)

assert adata.obs["predicted_doublet"].dtype == bool, (
    "predicted_doublet was not stored as an ordinary boolean column."
)


# ------------------------------------------------------------------
# Report completion
# ------------------------------------------------------------------
n_predicted_doublets = int(
    adata.obs["predicted_doublet"].sum()
)

predicted_doublet_fraction = (
    n_predicted_doublets / adata.n_obs
)

print()
print("Scrublet completed successfully.")
print("Cells scored:", int(adata.n_obs))
print("Predicted doublets:", n_predicted_doublets)
print(
    "Predicted doublet fraction:",
    f"{predicted_doublet_fraction:.4%}",
)

adata.obs[
    ["doublet_score", "predicted_doublet"]
].head()

Scrublet input shape: (23207, 29457)
Scrublet input matrix: csr_matrix
Scrublet matrix dtype: int32
Scrublet matrix nonzero entries: 30643240
Scrublet parameters used:
  batch_key: None
  sim_doublet_ratio: 2.0
  expected_doublet_rate: 0.05
  stdev_doublet_rate: 0.02
  synthetic_doublet_umi_subsampling: 1.0
  knn_dist_metric: euclidean
  normalize_variance: True
  log_transform: False
  mean_center: False
  n_prin_comps: 30
  use_approx_neighbors: True
  get_doublet_neighbor_parents: False
  n_neighbors: None
  threshold: None
  verbose: True
  copy: False
  random_state: 0

Scrublet completed successfully.
Cells scored: 23207
Predicted doublets: 6
Predicted doublet fraction: 0.0259%


,doublet_score,predicted_doublet
cell_id,,
PJ016_cell000216,0.030897,False
PJ016_cell000217,0.037320,False
PJ016_cell000218,0.066561,False
PJ016_cell000219,0.154259,False
PJ016_cell000220,0.034826,False


In [6]:
per_sample_summary = (
    adata.obs.groupby("sample_id", observed=True)
    .agg(
        cells=("predicted_doublet", "size"),
        predicted_doublets=("predicted_doublet", "sum"),
        mean_doublet_score=("doublet_score", "mean"),
        median_doublet_score=("doublet_score", "median"),
        maximum_doublet_score=("doublet_score", "max"),
    )
    .reset_index()
)
per_sample_summary["predicted_doublet_fraction"] = (
    per_sample_summary["predicted_doublets"]
    / per_sample_summary["cells"]
)
per_sample_summary.to_csv(
    AUDIT_DIR / "GSE103224_scrublet_summary_by_sample.tsv",
    sep="\t",
    index=False,
)

score_table = adata.obs[
    ["sample_id", "doublet_score", "predicted_doublet"]
].copy()
score_table.to_csv(
    AUDIT_DIR / "GSE103224_scrublet_per_cell_scores.tsv.gz",
    sep="\t",
    compression="gzip",
)

per_sample_summary

,sample_id,cells,predicted_doublets,mean_doublet_score,median_doublet_score,maximum_doublet_score,predicted_doublet_fraction
0,PJ016,2838,0,0.042800,0.036677,0.392540,0.000000
1,PJ017,1237,0,0.059806,0.051581,0.422053,0.000000
2,PJ018,2163,0,0.047219,0.035431,0.456033,0.000000
3,PJ025,5829,5,0.042148,0.033650,0.665689,0.000858
4,PJ030,3048,0,0.032942,0.022001,0.495575,0.000000
5,PJ032,1354,0,0.042613,0.030897,0.542169,0.000000
6,PJ035,3707,1,0.035968,0.026079,0.665689,0.000270
7,PJ048,3031,0,0.035032,0.026525,0.542169,0.000000


In [7]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(
    adata.obs["doublet_score"].to_numpy(),
    bins=60,
    edgecolor="black",
)
ax.set_xlabel("Scrublet doublet score")
ax.set_ylabel("Cells")
ax.set_title("GSE103224 post-QC Scrublet scores")
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "GSE103224_scrublet_score_histogram.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close(fig)

In [8]:
predicted_count = int(adata.obs["predicted_doublet"].sum())

doublet_summary = {
    "dataset_id": "DS1_GSE103224",
    "geo_accession": "GSE103224",
    "scope": "complete post-QC DS1 object",
    "method": "scanpy.pp.scrublet",
    "parameters_requested": REQUESTED_SCRUBLET_PARAMETERS,
    "parameters_used": scrublet_parameters,
    "input_cells": int(adata.n_obs),
    "input_genes": int(adata.n_vars),
    "predicted_doublets": predicted_count,
    "predicted_doublet_fraction": float(predicted_count / adata.n_obs),
    "doublets_removed": False,
    "output_cells": int(adata.n_obs),
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "output_h5ad": str(OUTPUT_H5AD),
}

# A separate key avoids modifying Scanpy's own Scrublet metadata.
adata.uns["doublet_detection_provenance"] = {
    "scope": "post_QC_dataset_level_diagnostic",
    "method": "scanpy.pp.scrublet",
    "doublets_removed": "false",
    "predicted_doublets": predicted_count,
    "parameters_json": json.dumps(scrublet_parameters, sort_keys=True),
}

with open(
    AUDIT_DIR / "GSE103224_scrublet_summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(doublet_summary, handle, indent=2)

print(doublet_summary)

{'dataset_id': 'DS1_GSE103224', 'geo_accession': 'GSE103224', 'scope': 'complete post-QC DS1 object', 'method': 'scanpy.pp.scrublet', 'parameters_requested': {'sim_doublet_ratio': 2.0, 'expected_doublet_rate': 0.05, 'n_prin_comps': 30, 'random_state': 0}, 'parameters_used': {'batch_key': None, 'sim_doublet_ratio': 2.0, 'expected_doublet_rate': 0.05, 'stdev_doublet_rate': 0.02, 'synthetic_doublet_umi_subsampling': 1.0, 'knn_dist_metric': 'euclidean', 'normalize_variance': True, 'log_transform': False, 'mean_center': False, 'n_prin_comps': 30, 'use_approx_neighbors': True, 'get_doublet_neighbor_parents': False, 'n_neighbors': None, 'threshold': None, 'verbose': True, 'copy': False, 'random_state': 0}, 'input_cells': 23207, 'input_genes': 29457, 'predicted_doublets': 6, 'predicted_doublet_fraction': 0.0002585426810876029, 'doublets_removed': False, 'output_cells': 23207, 'timestamp': '2026-08-03T15:16:20', 'output_h5ad': 'C:\\Users\\AmirSilco\\Documents\\GitHub\\glioma-cnv-single-cell-atl

In [9]:
if adata.shape != EXPECTED_INPUT_SHAPE:
    raise AssertionError(
        "The doublet diagnostic unexpectedly changed the object shape: "
        f"{adata.shape}"
    )

temporary_output = OUTPUT_H5AD.with_suffix(".tmp.h5ad")
if temporary_output.exists():
    temporary_output.unlink()

adata.write_h5ad(temporary_output, compression="gzip")
temporary_output.replace(OUTPUT_H5AD)

check = sc.read_h5ad(OUTPUT_H5AD, backed="r")
try:
    if check.shape != EXPECTED_INPUT_SHAPE:
        raise AssertionError(
            f"Saved-file validation failed: observed {check.shape}"
        )
    for column in ("doublet_score", "predicted_doublet"):
        if column not in check.obs:
            raise AssertionError(
                f"Saved output is missing adata.obs[{column!r}]."
            )
finally:
    check.file.close()

print("Saved and validated:", OUTPUT_H5AD)

Saved and validated: C:\Users\AmirSilco\Documents\GitHub\glioma-cnv-single-cell-atlas\data\processed\DS1_GSE103224\GSE103224_postQC_scrublet_checked.h5ad
